# Stationarity analysis

This notebook demonstrates the `nolitisea.stationarity` module.  Nonlinear
tools such as dimension or Lyapunov estimators assume that the dynamics does
not change over the observation interval, so a first sanity check on any
series is whether it is stationary.  The module provides three classical
instruments:

| # | Routine | Purpose | Returns |
|---|---------|---------|---------|
| 1 | `recurrence.recurr` / `recurrence_matrix` + RQA metrics | recurrence plot and its quantifiers | pair list / boolean matrix, RR, DET, LAM, ... |
| 2 | `stp.stp` | space-time separation plot (Theiler window choice) | distance quantiles per time shift |
| 3 | `nstat_z.nstat_z` | cross-prediction nonstationarity test | segment-to-segment normalised forecast-error matrix |

The running example is the x-component of a Lorenz trajectory for the
recurrence and space-time analyses (approximately stationary), plus two
synthetic series — stationary white noise and a piecewise mean-shifted noise —
whose cross-prediction matrices make the contrast explicit.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from nolitisea.generate.lorenz import lorenz
from nolitisea.stationarity.nstat_z import nstat_z
from nolitisea.stationarity.recurrence import (
    average_diagonal_length,
    determinism,
    entropy_diagonal_lines,
    laminarity,
    recurr,
    recurrence_matrix,
    recurrence_rate,
    trapping_time,
)
from nolitisea.stationarity.stp import stp

# Lorenz trajectory sampled at dt = 0.03; analyse the x-component.
t, states = lorenz(length=6000)
x = states[:, 0]

print(f"states: shape {states.shape}, dt = {t[1] - t[0]:.3f}")
print(f"x: range [{x.min():.3f}, {x.max():.3f}]")

## 1. `recurr` — recurrence plot

`recurr` builds the recurrence plot: every component of the
series is rescaled to `[0, 1]`, the delay embedding is built, and each pair of
embedding vectors whose **Chebyshev** distance is below the threshold `eps`
is reported as a recurrence.  The result is the upper triangle of the
symmetric recurrence matrix, returned as an `(N, 2)` array of row indices.
`eps` is expressed in the units of the input data (`None` uses the data
interval divided by 1000).

In [ ]:
rec = recurr(x, embed=3, delay=10, eps=2.0)
pairs = rec["pairs"]
n_pts = rec["n_points"]

print(f"embedding points : {n_pts}")
print(f"eps (rescaled)   : {rec['eps']:.4f}")
print(f"recurrence pairs : {pairs.shape[0]} (upper triangle)")
print(f"recurrence rate  : {2 * pairs.shape[0] / (n_pts * (n_pts - 1)):.4f}")

### Recurrence plot

Each pair `(i, j)` is one dark pixel at position `(i, j)` (and its mirror
`(j, i)`).  Diagonal lines correspond to segments of the trajectory that
recur after some time; vertical/horizontal lines mark times where the orbit
lingers near the same state.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(pairs[:, 0], pairs[:, 1], s=0.3, color="steelblue", rasterized=True)
ax.set_xlim(0, n_pts)
ax.set_ylim(0, n_pts)
ax.set_xlabel("t")
ax.set_ylabel("t'")
ax.set_title("Recurrence plot (Lorenz x, embed=3, delay=10)")
plt.tight_layout()
plt.show()

### RQA quantifiers via `recurrence_matrix`

For small-to-medium series the full boolean recurrence matrix
`recurrence_matrix(series, dim, delay, eps)` fits in memory, and the
classical recurrence quantification analysis (RQA) metrics can be computed
from it directly: recurrence rate `RR`, determinism `DET` (fraction of
recurrence points on diagonal lines), average length and Shannon entropy of
the diagonal lines, laminarity `LAM` and trapping time `TT` (vertical
structures).

In [ ]:
# A 1500-point segment keeps the full (1500, 1500) distance matrix small.
seg = x[:1500]
R = recurrence_matrix(seg, dim=3, delay=10, eps=5.0)

metrics = {
    "RR": recurrence_rate(R),
    "DET": determinism(R),
    "L_avg": average_diagonal_length(R),
    "ENTR": entropy_diagonal_lines(R),
    "LAM": laminarity(R),
    "TT": trapping_time(R),
}
for name, value in metrics.items():
    print(f"{name:5s} = {value:.4f}")

## 2. `stp` — space-time separation plot

The space-time separation plot tracks how the distance between
embedding vectors grows with their temporal separation.  For every time shift
`t` it collects the Chebyshev distances between each vector and its
`t`-shifted copy and reports, for each cumulative fraction level, the
distance below which that fraction of pairs lies.  The curves rise for small
`t` and saturate once the shift exceeds the correlation time — the plateau
onset is the standard choice for the **Theiler window** used to exclude
temporally correlated neighbours in neighbour-based estimators.

In [ ]:
sep = stp(x, dim=3, delay=10, max_time=150, fraction=0.05)

print(f"fraction levels : {sep['fractions']}")
print(f"curve matrix    : {sep['stp'].shape} (fractions x time shifts)")

### Separation curves

One curve per cumulative fraction: at shift `t` the curve value is the
distance below which that fraction of vector pairs lies.  All curves flatten
after a few correlation times — shifts beyond the plateau are safe choices
for a Theiler window.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
show = [0, 4, 9, 14, 19]  # fractions 0.05, 0.25, 0.50, 0.75, 1.00
for idx in show:
    ax.plot(sep["times"], sep["stp"][idx],
            label=f"fraction {sep['fractions'][idx]:.2f}")
ax.set_xlabel("time shift t")
ax.set_ylabel("Chebyshev separation")
ax.set_title("Space-time separation plot (Lorenz x)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. `nstat_z` — cross-prediction nonstationarity test

`nstat_z` splits the series into `n_pieces` contiguous segments, builds a
zeroth-order neighbour predictor from each source segment and forecasts every
other segment with it.  `matrix[first, second]` is the forecast error of
predicting segment `second` from segment `first`, normalised by the standard
deviation of segment `second`:

- **stationary series** → all entries ≈ 1 (every segment predicts every
  other segment equally well, including itself);
- **nonstationary series** → diagonal valley (self-prediction is much better
  than cross-prediction between different dynamical regimes).

In [ ]:
rng = np.random.default_rng(42)

# (a) Stationary white noise.
w = rng.standard_normal(3000)
res_stat = nstat_z(w, dim=3, delay=1, n_pieces=6, min_neighbors=30)
m_stat = res_stat["matrix"]

print("stationary white noise:")
print(np.round(m_stat, 3))
print(f"coefficient of variation = {m_stat.std() / m_stat.mean():.4f}")

### Stationary vs nonstationary contrast

The second series concatenates four Gaussian segments whose mean shifts by 2
per segment.  The cross-prediction error between segments grows with their
mean separation, producing the staircase pattern in the matrix, while the
diagonal stays near 1.

In [ ]:
# (b) Piecewise mean-shifted noise.
y_ns = np.concatenate([rng.standard_normal(1000) + mu for mu in (0, 2, 4, 6)])
res_ns = nstat_z(y_ns, dim=3, delay=1, n_pieces=4, min_neighbors=30)
m_ns = res_ns["matrix"]

print("nonstationary (mean-shifted segments):")
print(np.round(m_ns, 3))
print(f"mean diagonal     = {np.diag(m_ns).mean():.3f}")
print(f"mean off-diagonal = {m_ns[~np.eye(4, dtype=bool)].mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, m, title in (
    (axes[0], m_stat, "stationary white noise"),
    (axes[1], m_ns, "piecewise mean-shifted noise"),
):
    im = ax.imshow(m, origin="lower", cmap="viridis")
    ax.set_xlabel("source segment (first)")
    ax.set_ylabel("target segment (second)")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="normalised RMSE")
plt.tight_layout()
plt.show()

## Summary

`nolitisea.stationarity` covers the standard stationarity workflow:

| Task | Routine | Result on the examples |
|------|---------|------------------------|
| recurrence plot | `recurr` | Lorenz x: diagonal lines from recurring orbit segments |
| RQA quantifiers | `recurrence_matrix` + metrics | `DET` well above `RR` → deterministic dynamics |
| Theiler window | `stp` | curves saturate after a few tens of shifts |
| stationarity test | `nstat_z` | white noise → uniform matrix; mean-shifted noise → diagonal valley |